## 7. Conclusiones

### Hallazgos principales

1. **El tabaquismo es el factor dominante:** Ser fumador incrementa el costo del seguro en **$23,849 anuales**, lo que representa aproximadamente 3.8 veces mas que un no fumador. Este es, por mucho, el predictor mas fuerte del modelo (t-stat = 57.7).

2. **Edad e IMC son predictores significativos:** Cada ano adicional agrega ~$257 al costo, y cada punto de IMC agrega ~$339. Ambas variables son altamente significativas (p < 0.001).

3. **El sexo no influye significativamente:** La diferencia entre hombres y mujeres no es estadisticamente significativa (p = 0.69), lo que sugiere que las aseguradoras no deberian basar las primas en el genero.

4. **Las regiones del sureste y suroeste tienen costos ligeramente menores** que el noreste (categoria de referencia), con diferencias estadisticamente significativas.

### Rendimiento del modelo
- El modelo explica el **75.1%** de la variabilidad en los costos (R² = 0.7509).
- El R² ajustado (0.7494) confirma que los predictores son genuinamente utiles y no hay sobreajuste.
- El analisis de residuos revela cierta heterocedasticidad, lo que sugiere que un modelo no lineal o con interacciones podria mejorar las predicciones.

### Posibles mejoras futuras
- Incorporar **interacciones** (e.g., `smoker x bmi`) para capturar efectos no lineales
- Probar modelos como **Random Forest** o **Gradient Boosting** para comparar rendimiento
- Implementar **validacion cruzada** para evaluar la generalizacion del modelo

---
*Proyecto desarrollado implementando regresion lineal desde cero con algebra matricial (NumPy), sin depender de sklearn.*

In [ ]:
# Visualizacion de la magnitud de los coeficientes (excluyendo intercepto)
coef_df = pd.DataFrame({
    'Variable': nombres[1:],
    'Coeficiente': betas[1:],
    'p_value': p_values[1:],
    'Significativo': ['Si' if p < 0.05 else 'No' for p in p_values[1:]]
}).sort_values('Coeficiente', key=abs, ascending=True)

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#2ecc71' if sig == 'Si' else '#95a5a6' for sig in coef_df['Significativo']]
bars = ax.barh(coef_df['Variable'], coef_df['Coeficiente'], color=colors, edgecolor='white', height=0.6)

ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_title('Impacto de cada Variable en el Costo del Seguro', fontsize=14, fontweight='bold')
ax.set_xlabel('Cambio en Cargos ($) por Unidad de Cambio', fontsize=12)

# Agregar valores en las barras
for bar, val in zip(bars, coef_df['Coeficiente']):
    x_pos = bar.get_width()
    ax.text(x_pos + 300 if x_pos > 0 else x_pos - 300, bar.get_y() + bar.get_height()/2,
            f'${val:,.0f}', ha='left' if x_pos > 0 else 'right', va='center', fontsize=10, fontweight='bold')

ax.legend(['Verde = Significativo (p<0.05)', 'Gris = No significativo'], loc='lower right', fontsize=10)
plt.tight_layout()
plt.show()

### 6.4 Importancia relativa de las variables

# Analisis Predictivo de Costos de Seguros Medicos en Estados Unidos

## Regresion Lineal Multiple desde Cero

En este proyecto se construye un modelo de regresion lineal multiple **desde cero** (sin usar scikit-learn) para predecir los costos de seguros medicos en Estados Unidos. El objetivo es comprender como factores demograficos y de estilo de vida influyen en el precio de las primas de seguro.

**Aspectos clave del analisis:**
- Exploracion y visualizacion de un dataset con 1,338 registros
- Codificacion manual de variables categoricas (one-hot encoding)
- Implementacion de regresion lineal usando algebra matricial (metodo de minimos cuadrados ordinarios)
- Evaluacion estadistica completa: R², errores estandar, t-statistics y p-values

**Dataset:** [Medical Cost Personal Dataset](https://www.kaggle.com/mirichoi0218/insurance) con informacion de edad, sexo, IMC, hijos, habito de fumar, region y cargos medicos.

## 1. Importacion de Librerias y Carga de Datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Configuracion de visualizacion
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('coolwarm')
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['font.size'] = 12

print("Librerias cargadas correctamente")

In [ ]:
df = pd.read_csv('insurance.csv')
print(f"Dataset cargado: {df.shape[0]} registros, {df.shape[1]} variables\n")
df.head(10)

## 2. Exploracion Inicial del Dataset

Antes de modelar, es fundamental comprender la estructura de los datos: tipos de variables, valores faltantes y distribucion de cada feature.

In [ ]:
print("=" * 55)
print("INFORMACION GENERAL DEL DATASET")
print("=" * 55)
print(f"\nDimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
print(f"\nTipos de datos:")
print(df.dtypes.to_string())
print(f"\nValores faltantes:")
print(df.isnull().sum().to_string())
print(f"\nTotal valores faltantes: {df.isnull().sum().sum()}")

In [ ]:
print("=" * 55)
print("ESTADISTICAS DESCRIPTIVAS - VARIABLES NUMERICAS")
print("=" * 55)
df.describe().round(2)

In [ ]:
print("=" * 55)
print("DISTRIBUCION DE VARIABLES CATEGORICAS")
print("=" * 55)
for col in ['sex', 'smoker', 'region']:
    print(f"\n{col.upper()}:")
    counts = df[col].value_counts()
    for val, count in counts.items():
        pct = count / len(df) * 100
        print(f"  {val}: {count} ({pct:.1f}%)")

## 3. Analisis Exploratorio de Variables (EDA)

Visualizamos la distribucion de las variables principales y su relacion con el costo del seguro (`charges`) para identificar patrones antes de construir el modelo.

In [ ]:
# Distribucion de la variable objetivo: charges
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histograma
axes[0].hist(df['charges'], bins=40, color='#3498db', edgecolor='white', alpha=0.85)
axes[0].axvline(df['charges'].mean(), color='#e74c3c', linestyle='--', linewidth=2, label=f"Media: ${df['charges'].mean():,.0f}")
axes[0].axvline(df['charges'].median(), color='#2ecc71', linestyle='--', linewidth=2, label=f"Mediana: ${df['charges'].median():,.0f}")
axes[0].set_title('Distribucion de Costos de Seguro', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Cargos ($)', fontsize=12)
axes[0].set_ylabel('Frecuencia', fontsize=12)
axes[0].legend(fontsize=11)

# Boxplot
sns.boxplot(y=df['charges'], ax=axes[1], color='#3498db', width=0.4)
axes[1].set_title('Boxplot de Costos de Seguro', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Cargos ($)', fontsize=12)

plt.tight_layout()
plt.show()

print(f"La distribucion presenta sesgo positivo (skewness: {df['charges'].skew():.2f})")
print(f"Esto indica que la mayoria de los asegurados pagan menos de ${df['charges'].median():,.0f},")
print(f"pero existe un grupo con costos significativamente mas altos.")

In [ ]:
# Relacion entre variables numericas y charges
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Edad vs Charges
axes[0].scatter(df['age'], df['charges'], alpha=0.5, c=df['smoker'].map({'yes': '#e74c3c', 'no': '#3498db'}), s=20)
axes[0].set_title('Edad vs Cargos', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Edad', fontsize=12)
axes[0].set_ylabel('Cargos ($)', fontsize=12)

# BMI vs Charges
axes[1].scatter(df['bmi'], df['charges'], alpha=0.5, c=df['smoker'].map({'yes': '#e74c3c', 'no': '#3498db'}), s=20)
axes[1].set_title('IMC vs Cargos', fontsize=14, fontweight='bold')
axes[1].set_xlabel('IMC (kg/m²)', fontsize=12)
axes[1].set_ylabel('Cargos ($)', fontsize=12)

# Children vs Charges
sns.boxplot(x='children', y='charges', data=df, ax=axes[2], palette='coolwarm')
axes[2].set_title('Numero de Hijos vs Cargos', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Numero de Hijos', fontsize=12)
axes[2].set_ylabel('Cargos ($)', fontsize=12)

plt.tight_layout()
plt.show()

# Leyenda manual
print("Color: Rojo = Fumador | Azul = No fumador")
print(f"\nCorrelacion edad-cargos: {df['age'].corr(df['charges']):.3f}")
print(f"Correlacion IMC-cargos: {df['bmi'].corr(df['charges']):.3f}")

In [ ]:
# Impacto de variables categoricas en los cargos
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Fumador vs No fumador
sns.boxplot(x='smoker', y='charges', data=df, ax=axes[0], palette=['#3498db', '#e74c3c'])
axes[0].set_title('Fumador vs Cargos', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Fumador', fontsize=12)
axes[0].set_ylabel('Cargos ($)', fontsize=12)

# Sexo
sns.boxplot(x='sex', y='charges', data=df, ax=axes[1], palette=['#9b59b6', '#1abc9c'])
axes[1].set_title('Sexo vs Cargos', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Sexo', fontsize=12)
axes[1].set_ylabel('Cargos ($)', fontsize=12)

# Region
sns.boxplot(x='region', y='charges', data=df, ax=axes[2], palette='Set2')
axes[2].set_title('Region vs Cargos', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Region', fontsize=12)
axes[2].set_ylabel('Cargos ($)', fontsize=12)

plt.tight_layout()
plt.show()

# Costo promedio por grupo
print("COSTO PROMEDIO POR GRUPO:")
print(f"  Fumadores: ${df[df['smoker']=='yes']['charges'].mean():,.0f}")
print(f"  No fumadores: ${df[df['smoker']=='no']['charges'].mean():,.0f}")
print(f"  Diferencia: {df[df['smoker']=='yes']['charges'].mean() / df[df['smoker']=='no']['charges'].mean():.1f}x mas caro")

In [ ]:
# Matriz de correlacion
fig, ax = plt.subplots(figsize=(10, 8))

# Crear version numerica para correlacion
df_numeric = df.copy()
df_numeric['sex_male'] = (df['sex'] == 'male').astype(int)
df_numeric['smoker_yes'] = (df['smoker'] == 'yes').astype(int)
df_numeric = df_numeric.drop(columns=['sex', 'smoker', 'region'])

corr_matrix = df_numeric.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            mask=mask, square=True, linewidths=0.5, ax=ax,
            cbar_kws={"shrink": 0.8})
ax.set_title('Matriz de Correlacion entre Variables', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

print("La variable 'smoker' muestra la correlacion mas fuerte con los cargos,")
print("confirmando que fumar es el principal predictor del costo del seguro.")

## 4. Preprocesamiento: Codificacion de Variables Categoricas

Para incluir variables categoricas en el modelo de regresion, se realiza una codificacion manual (one-hot encoding). La variable `region` se convierte en 3 variables dummy (dejando `northeast` como categoria de referencia para evitar multicolinealidad perfecta).

In [ ]:
# Variable objetivo
y = df['charges'].values

# Variables numericas
age = df['age'].values
bmi = df['bmi'].values
children = df['children'].values

# Variables categoricas (codificacion binaria)
sex_male = (df['sex'] == 'male').astype(int).values
smoker_yes = (df['smoker'] == 'yes').astype(int).values

# Region (3 dummies, referencia: northeast)
region_nw = (df['region'] == 'northwest').astype(int).values
region_se = (df['region'] == 'southeast').astype(int).values
region_sw = (df['region'] == 'southwest').astype(int).values

# Construir matriz de diseno X con columna de intercepto
n = len(y)
ones = np.ones(n)
X = np.column_stack([ones, age, bmi, children, sex_male, smoker_yes, region_nw, region_se, region_sw])

nombres = ['intercepto', 'age', 'bmi', 'children', 'sex_male', 
           'smoker_yes', 'region_nw', 'region_se', 'region_sw']

print(f"Matriz X: {X.shape[0]} observaciones x {X.shape[1]} variables (incluyendo intercepto)")
print(f"Vector y: {y.shape[0]} observaciones")

## 5. Modelo de Regresion Lineal Multiple (OLS desde Cero)

Se implementa el metodo de **Minimos Cuadrados Ordinarios (OLS)** usando algebra matricial. La formula cerrada para estimar los coeficientes es:

$$\hat{\beta} = (X^TX)^{-1}X^Ty$$

Este enfoque permite entender exactamente como funciona la regresion lineal internamente, sin depender de librerias de Machine Learning.

In [ ]:
# Implementacion OLS paso a paso
X_transpose = X.T                      # Transpuesta de X
XtX = X_transpose @ X                  # Producto X'X
XtX_inv = np.linalg.inv(XtX)          # Inversa de (X'X)
Xty = X_transpose @ y                  # Producto X'y
betas = XtX_inv @ Xty                  # Coeficientes estimados

print("=" * 60)
print("COEFICIENTES DEL MODELO DE REGRESION")
print("=" * 60)
for nombre, beta in zip(nombres, betas):
    if nombre == 'intercepto':
        print(f"  {nombre:<15} = ${beta:>12,.2f}")
    else:
        print(f"  {nombre:<15} = ${beta:>12,.2f}")

### Interpretacion de coeficientes clave:
- **smoker_yes (+$23,849):** Ser fumador incrementa el costo del seguro en casi $24,000 anuales. Es el factor mas determinante.
- **age (+$257 por ano):** Cada ano adicional de edad incrementa el costo en ~$257.
- **bmi (+$339 por unidad):** Cada punto adicional de IMC agrega ~$339 al costo.
- **children (+$476 por hijo):** Cada hijo adicional incrementa el costo en ~$476.
- **sex_male (-$131):** El sexo tiene un efecto minimo y probablemente no significativo.

## 6. Evaluacion del Modelo

### 6.1 Bondad de ajuste (R² y R² ajustado)

In [ ]:
# Valores predichos y metricas de ajuste
y_pred = X @ betas
y_mean = np.mean(y)

# R cuadrado
SS_res = np.sum((y - y_pred) ** 2)
SS_tot = np.sum((y - y_mean) ** 2)
R2 = 1 - (SS_res / SS_tot)

# R cuadrado ajustado
k = X.shape[1] - 1  # numero de predictores
R2_adj = 1 - ((1 - R2) * (n - 1)) / (n - k - 1)

# Error cuadratico medio
MSE = SS_res / n
RMSE = np.sqrt(MSE)

print("=" * 55)
print("METRICAS DE BONDAD DE AJUSTE")
print("=" * 55)
print(f"  R²:          {R2:.4f}")
print(f"  R² ajustado: {R2_adj:.4f}")
print(f"  RMSE:        ${RMSE:,.2f}")
print(f"\n  El modelo explica el {R2*100:.1f}% de la variacion")
print(f"  en los costos de seguro medico.")

### 6.2 Inferencia estadistica: errores estandar, t-statistics y p-values

Para determinar la significancia estadistica de cada coeficiente, se calculan los errores estandar, los estadisticos t y los p-values. Un p-value < 0.05 indica que el coeficiente es estadisticamente significativo.

In [ ]:
# Inferencia estadistica
residuos = y - y_pred
gl = n - k - 1  # grados de libertad
sigma2 = SS_res / gl

# Errores estandar
var_betas = sigma2 * np.diag(XtX_inv)
se_betas = np.sqrt(var_betas)

# t-statistics y p-values
t_stats = betas / se_betas
p_values = 2 * (1 - stats.t.cdf(np.abs(t_stats), gl))

# Tabla resumen completa
print("=" * 75)
print("RESUMEN ESTADISTICO DEL MODELO OLS")
print("=" * 75)
print(f"{'Variable':<15} {'Coef.':>12} {'Std Err':>12} {'t-stat':>10} {'P>|t|':>10} {'Sig.':>5}")
print("-" * 75)
for i in range(len(nombres)):
    sig = "***" if p_values[i] < 0.001 else "**" if p_values[i] < 0.01 else "*" if p_values[i] < 0.05 else ""
    print(f"{nombres[i]:<15} {betas[i]:>12.2f} {se_betas[i]:>12.2f} {t_stats[i]:>10.3f} {p_values[i]:>10.4f} {sig:>5}")
print("-" * 75)
print(f"{'R²:':<15} {R2:.4f}    {'R² ajustado:':<15} {R2_adj:.4f}")
print(f"{'Observaciones:':<15} {n}      {'Grados libertad:':<15} {gl}")
print("=" * 75)
print("\nSignificancia: *** p<0.001  ** p<0.01  * p<0.05")

### 6.3 Analisis de residuos

In [ ]:
# Analisis visual de residuos
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Residuos vs Valores predichos
axes[0].scatter(y_pred, residuos, alpha=0.4, s=15, color='#3498db')
axes[0].axhline(y=0, color='#e74c3c', linestyle='--', linewidth=1.5)
axes[0].set_title('Residuos vs Valores Predichos', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Valores Predichos ($)', fontsize=11)
axes[0].set_ylabel('Residuos ($)', fontsize=11)

# Distribucion de residuos
axes[1].hist(residuos, bins=40, color='#3498db', edgecolor='white', alpha=0.85, density=True)
x_range = np.linspace(residuos.min(), residuos.max(), 100)
axes[1].plot(x_range, stats.norm.pdf(x_range, residuos.mean(), residuos.std()), 
             color='#e74c3c', linewidth=2, label='Distribucion Normal')
axes[1].set_title('Distribucion de Residuos', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Residuos ($)', fontsize=11)
axes[1].set_ylabel('Densidad', fontsize=11)
axes[1].legend()

# Real vs Predicho
axes[2].scatter(y, y_pred, alpha=0.4, s=15, color='#3498db')
axes[2].plot([y.min(), y.max()], [y.min(), y.max()], color='#e74c3c', linestyle='--', linewidth=2, label='Prediccion perfecta')
axes[2].set_title('Valores Reales vs Predichos', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Cargos Reales ($)', fontsize=11)
axes[2].set_ylabel('Cargos Predichos ($)', fontsize=11)
axes[2].legend()

plt.tight_layout()
plt.show()

print(f"Media de residuos: ${np.mean(residuos):.6f} (esperado: ~0)")
print(f"Desviacion estandar de residuos: ${np.std(residuos):,.2f}")